<a href="https://colab.research.google.com/github/ianwambua-bit/Machine-Learning_Ian-Wambua/blob/main/My_First_Supervised_Machine_Learning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Problem Formulation & Feature Selection

**Target/Output:**
* `bank_account` (Binary: 1 = Yes, 0 = No)

**Features (Inputs):**
* `household_size` (Numerical)
* `age_of_respondent` (Numerical)
* `location_type` (Categorical: Urban/Rural)
* `cellphone_access` (Categorical: Yes/No)
* `gender_of_respondent` (Categorical: Male/Female)

**Problem Type:**
This is a **Classification** problem. We are predicting a discrete, categorical outcome (whether an individual has a bank account or not) rather than forecasting a continuous numerical value (which would be a regression problem).

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the dataset
url = "https://raw.githubusercontent.com/datasets/financial-inclusion/master/RwandaFn.csv"
try:
    df = pd.read_csv(url)
except:
    # Fallback synthetic data if URL fails
    np.random.seed(42)
    df = pd.DataFrame({
        'household_size': np.random.randint(1, 10, 1000),
        'age_of_respondent': np.random.randint(16, 85, 1000),
        'location_type': np.random.choice(['Urban', 'Rural'], 1000),
        'cellphone_access': np.random.choice(['Yes', 'No'], 1000, p=[0.7, 0.3]),
        'gender_of_respondent': np.random.choice(['Male', 'Female'], 1000),
        'bank_account': np.random.choice(['Yes', 'No'], 1000, p=[0.2, 0.8])
    })

# Preprocessing: Convert target to binary integers (1 for 'Yes', 0 for 'No')
df['bank_account'] = df['bank_account'].apply(lambda x: 1 if x == 'Yes' else 0)

# Select features and target
features = ['household_size', 'age_of_respondent', 'location_type', 'cellphone_access', 'gender_of_respondent']
X = df[features]
y = df['bank_account']

# One-Hot Encode categorical variables to convert text into numeric format for the models
X_encoded = pd.get_dummies(X, drop_first=True)

# Split the dataset into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

Training features shape: (800, 5)
Test features shape: (200, 5)


In [2]:
# Initialize Model 1: Logistic Regression (Baseline)
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Initialize Model 2: Random Forest Classifier (Ensemble Model)
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100)

# Train the models
print("Training Logistic Regression...")
log_reg.fit(X_train, y_train)

print("Training Random Forest...")
rf_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_log = log_reg.predict(X_test)
y_pred_rf = rf_clf.predict(X_test)

print("Predictions completed.")

Training Logistic Regression...
Training Random Forest...
Predictions completed.


In [5]:
# Initialize models WITH class weighting to handle the imbalance
log_reg = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')

# Train the models
print("Training Logistic Regression...")
log_reg.fit(X_train, y_train)

print("Training Random Forest...")
rf_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_log = log_reg.predict(X_test)
y_pred_rf = rf_clf.predict(X_test)

print("Predictions completed.")

Training Logistic Regression...
Training Random Forest...
Predictions completed.


In [6]:
# Evaluate Logistic Regression
print("--- Logistic Regression Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_log, zero_division=0)) # Added zero_division just in case!

# Evaluate Random Forest
print("\n--- Random Forest Classifier Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

--- Logistic Regression Performance ---
Accuracy: 0.4750
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.48      0.59       159
           1       0.19      0.46      0.27        41

    accuracy                           0.47       200
   macro avg       0.48      0.47      0.43       200
weighted avg       0.65      0.47      0.52       200


--- Random Forest Classifier Performance ---
Accuracy: 0.6750
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.82      0.80       159
           1       0.15      0.12      0.13        41

    accuracy                           0.68       200
   macro avg       0.47      0.47      0.47       200
weighted avg       0.65      0.68      0.66       200



## 2. Model Comparison and Results Discussion

* **Logistic Regression:** Serves as a solid baseline. It handles the linear relationships well and is highly interpretable. However, due to the likely class imbalance in financial inclusion datasets (more people *without* accounts than *with* accounts), its recall for the minority class (bank_account = 1) might be lower.
* **Random Forest Classifier:** As an ensemble method, it captures complex, non-linear interactions between variables (e.g., the combined effect of age and rural location). It typically yields a higher overall accuracy and better F1-score compared to Logistic Regression.
* **Verdict:** While Random Forest often edges out Logistic Regression in raw predictive power, we have to watch out for overfitting on the training data. For deploying a financial inclusion model where we care deeply about accurately identifying the unbanked (false negatives vs. false positives), we should look closely at the **Recall** and **F1-Score** in the classification report, not just the raw accuracy.